<!-- --8<-- encabezado-grupo
Archivo: notebooks/ezequiel_paso_a_paso.ipynb
Objetivo: Notebook: ezequiel_paso_a_paso
Grupo del curso: Fabio Durán - Gustavo Reyes - Ezequiel Muñoz
Creación: 20261110 1922
Última modificación: 20261110 1934
--8<-- fin-encabezado-grupo -->

# Agente de Ezequiel (IA Engineer & Data Scientist) · paso a paso

Tres pasos: **(1) herramientas**, **(2) el agente por A2A**, **(3) consumo**.

Pasos 2 y 3 requieren `uv run langgraph dev` corriendo y `GOOGLE_API_KEY` en `.env`.

## 1 · Herramientas (sin servidor ni API key)

Las tres herramientas de ML/Data Science con datos simulados.

In [ ]:
from src.agente_ezequiel.herramientas import recomendar_tecnica, metricas_recomendadas, elegir_modelo
print(recomendar_tecnica.invoke({'problema':'desbalance de clases'}))
print(metricas_recomendadas.invoke({'tipo_problema':'regresion'}))
print(elegir_modelo.invoke({'tipo_de_dato':'texto'}))

## 2 · El agente por A2A nativo

Con `langgraph dev` arriba, el servidor publica `agente_ezequiel`. Obtenemos su `assistant_id` y su tarjeta.

In [ ]:
import os, httpx
from dotenv import load_dotenv; load_dotenv()
BASE='http://127.0.0.1:2024'
aid=httpx.post(f'{BASE}/assistants/search', json={'graph_id':'agente_ezequiel'}, timeout=10).json()[0]['assistant_id']
print('assistant_id:', aid)
card=httpx.get(f'{BASE}/.well-known/agent-card.json', params={'assistant_id':aid}, timeout=10).json()
print('nombre:', card.get('name')); print('skills:', [s.get('name') for s in card.get('skills',[])])

## 3 · Consumo

Mandamos preguntas por `message/send`. El agente usa sus herramientas y rechaza lo que no es de su tema.

In [ ]:
import uuid, httpx
def preguntar(texto):
    cuerpo={'jsonrpc':'2.0','id':'1','method':'message/send','params':{'message':{'role':'user','messageId':uuid.uuid4().hex,'contextId':str(uuid.uuid4()),'parts':[{'kind':'text','text':texto}]}}}
    r=httpx.post(f'{BASE}/a2a/{aid}', json=cuerpo, timeout=90).json()
    print('>',texto); print(r['result'] if 'result' in r else r, '\n')
preguntar('¿Qué técnica uso si tengo desbalance de clases?')
preguntar('¿Qué métricas miro en un problema de regresión?')
preguntar('¿Cuál es la capital de Francia?')  # fuera de tema: lo rechaza